# TransformerConv

In [ ]:
!pip install torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.8 MB/s eta 0:00:00


In [ ]:
import os, pickle, itertools, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.data import DataLoader
from torch_geometric.nn import global_mean_pool, global_max_pool
from tqdm.notebook import tqdm
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Mounted at /content/drive
Using device: cuda


In [ ]:
BASE_PATH  = '/content/drive/MyDrive/team3_xai_gnn'
DATA_DIR   = os.path.join(BASE_PATH, 'preprocessed')
OUTPUT_DIR = os.path.join(BASE_PATH, 'transformer_results')
INNER_RESULTS_PATH = os.path.join(OUTPUT_DIR, 'inner_results.pkl')
os.makedirs(OUTPUT_DIR, exist_ok=True)

### Model definition

In [ ]:
class TransformerConvLayer(nn.Module):
    def __init__(self, node_dim, edge_dim, out_dim, heads=4):
        super().__init__()
        self.heads   = heads
        self.d       = out_dim // heads   # dimension per head

        # one linear projection per head, for Q, K, V and edge
        self.W_q = nn.Linear(node_dim, out_dim)
        self.W_k = nn.Linear(node_dim, out_dim)
        self.W_v = nn.Linear(node_dim, out_dim)
        self.W_e = nn.Linear(edge_dim,  out_dim)

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index    # source (sender), destination (receiver)

        Q = self.W_q(x)          # query
        K = self.W_k(x)          # key
        V = self.W_v(x)          # value
        E = self.W_e(edge_attr)  # embedding

        # --- attention score (one number per edge)
        # dot product of query of dst node with (key of src node + edge embedding)
        # dividing by sqrt(d) for numerical stability
        score = (Q[dst] * (K[src] + E)).sum(dim=-1) / (self.d ** 0.5)
        # shape: (num_edges,)

        # --- these are the final attention weights
        # for each destination node, softmax over all its incoming edges
        # if node i has m neighbors, their m scores get softmaxes into
        # m numbers that sum to 1
        score_max = torch.zeros(x.size(0), device=x.device).scatter_reduce_(0, dst, score, reduce='amax', include_self=True)
        score_exp = torch.exp(score - score_max[dst])
        denom = torch.zeros(x.size(0), device=x.device)
        denom.scatter_add_(0, dst, score_exp)
        alpha = score_exp / (denom[dst] + 1e-8)
        # shape: (num_edges,)

        # --- this is the actual message sent along each edge
        # the value of the sender plus the edge embedding
        # scaled by the attention weight
        msg = alpha.unsqueeze(-1) * (V[src] + E)
        # shape: (num_edges, hidden)

        # aggregate messages into destination nodes
        out = torch.zeros(x.size(0), msg.size(-1), device=x.device)
        out.scatter_add_(0, dst.unsqueeze(-1).expand_as(msg), msg)
        # shape: (num_nodes, hidden)

        return out

In [ ]:
class TransformerConvNet(nn.Module):
    def __init__(self, node_in, edge_in, hidden, heads, dropout):
        super().__init__()

        # --- input projection
        self.input_proj = nn.Linear(node_in, hidden)
        self.input_norm = nn.BatchNorm1d(hidden)

        # --- custom conv layers
        self.conv1 = TransformerConvLayer(hidden, edge_in, hidden, heads)
        self.norm1 = nn.BatchNorm1d(hidden)

        self.conv2 = TransformerConvLayer(hidden, edge_in, hidden, heads)
        self.norm2 = nn.BatchNorm1d(hidden)

        self.dropout = nn.Dropout(dropout)

        # --- classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2),
        )

    def forward(self, x, edge_index, edge_attr, batch):
        # --- input projection
        x = self.input_proj(x)
        x = self.input_norm(x)
        x = x.relu()

        # --- layer 1
        x = self.conv1(x, edge_index, edge_attr)
        x = self.norm1(x)
        x = x.relu()
        x = self.dropout(x)

        # --- layer 2
        x = self.conv2(x, edge_index, edge_attr)
        x = self.norm2(x)
        x = x.relu()
        x = self.dropout(x)

        # --- readout
        x_mean = global_mean_pool(x, batch)
        x_max  = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=-1)

        return self.classifier(x)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        # --- step 1
        # standard cross entropy, one value per sample (no reduction yet)
        # this gives us -log(pt) for each sample
        ce_loss = nn.functional.cross_entropy(
            logits, targets,
            weight=self.weight,
            reduction='none'   # keep per-sample losses, don't average yet
        )

        # --- step 2
        # recover pt (the probability assigned to the correct class)
        # ce_loss = -log(pt)  →  pt = exp(-ce_loss)
        pt = torch.exp(-ce_loss)

        # --- step 3
        # apply the focal modulating factor (1 - pt)^gamma
        # when pt is high (easy example, model is confident) → factor is small → loss shrinks
        # when pt is low  (hard example, model is uncertain) → factor is ~1   → loss unchanged
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()

### Helper functions for training and evaluating

In [ ]:
# helper functions
def get_labels(dataset):
    return np.array([d.y.item() for d in dataset])


def compute_metrics(labels, preds, probs):
    return {
        'balanced_acc':  balanced_accuracy_score(labels, preds),
        'accuracy':      accuracy_score(labels, preds),
        'recall_mci':    recall_score(labels, preds, pos_label=1, zero_division=0),
        'recall_cn':     recall_score(labels, preds, pos_label=0, zero_division=0),
        'precision_mci': precision_score(labels, preds, pos_label=1, zero_division=0),
        'precision_cn':  precision_score(labels, preds, pos_label=0, zero_division=0),
        'f1_mci':        f1_score(labels, preds, pos_label=1, zero_division=0),
        'f1_cn':         f1_score(labels, preds, pos_label=0, zero_division=0),
        'f1_macro':      f1_score(labels, preds, average='macro', zero_division=0),
        'auc':           roc_auc_score(labels, probs[:, 1]) if len(np.unique(labels)) > 1 else float('nan'),
    }


def avg_metrics(metrics_list):
    keys = metrics_list[0].keys()
    return {k: float(np.mean([m[k] for m in metrics_list])) for k in keys}


def passes_filter(avg, thresholds):
    return all(avg.get(k, 0.0) >= v for k, v in thresholds.items())


def build_criterion(cfg, train_labels):
    # weights are [1.0, mci_weight]
    weight = torch.tensor([1.0, cfg['mci_weight']], dtype=torch.float).to(device)
    return FocalLoss(gamma=cfg['gamma'], weight=weight)


def build_model_and_opt(cfg):
    model = TransformerConvNet(NODE_IN, EDGE_IN, HIDDEN, cfg['heads'], DROPOUT).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    return model, opt


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch  = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        loss   = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for batch in loader:
        batch  = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
        all_labels.append(batch.y.cpu().numpy())
        all_preds.append(probs.argmax(axis=1))
        all_probs.append(probs)
    return (
        np.concatenate(all_labels),
        np.concatenate(all_preds),
        np.concatenate(all_probs, axis=0),
    )


def train_with_early_stopping(model, opt, criterion, train_loader, val_loader):
    best_bal_acc   = -1.0
    best_state     = None
    best_metrics   = None
    patience_count = 0

    for epoch in range(EPOCHS):
        train_one_epoch(model, train_loader, criterion, opt)
        labels, preds, probs = evaluate(model, val_loader)
        metrics = compute_metrics(labels, preds, probs)

        if metrics['balanced_acc'] > best_bal_acc:
            best_bal_acc   = metrics['balanced_acc']
            best_state     = copy.deepcopy(model.state_dict())
            best_metrics   = metrics
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return model, best_metrics

### Defining parameters and different configurations

In [ ]:
# fixed params
NODE_IN  = 4
EDGE_IN  = 15
HIDDEN   = 32
DROPOUT  = 0.3
EPOCHS   = 50
PATIENCE = 15
BATCH    = 32
# k-folds
N_OUTER  = 5
N_INNER  = 3

# hyperparams for tuning
gammas        = [1.0, 1.4, 1.8]
mci_weights   = [3.7, 4.2]
lrs           = [1e-3, 3e-3]
heads_list    = [2, 4]
weight_decays = [5e-4, 1e-3]

CONFIGS = [
    {'mci_weight': mw, 'gamma': g, 'lr': lr, 'heads': h, 'wd': wd}
    for mw, g, lr, h, wd in itertools.product(mci_weights, gammas, lrs, heads_list, weight_decays)
]

print(f'Total configs: {len(CONFIGS)}')
for i, c in enumerate(CONFIGS):
    print(f'  [{i:02d}] {c}')

Total configs: 48
  [00] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.001, 'heads': 2, 'wd': 0.0005}
  [01] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.001, 'heads': 2, 'wd': 0.001}
  [02] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.001, 'heads': 4, 'wd': 0.0005}
  [03] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.001, 'heads': 4, 'wd': 0.001}
  [04] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.003, 'heads': 2, 'wd': 0.0005}
  [05] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.003, 'heads': 2, 'wd': 0.001}
  [06] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.003, 'heads': 4, 'wd': 0.0005}
  [07] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.003, 'heads': 4, 'wd': 0.001}
  [08] {'mci_weight': 3.7, 'gamma': 1.4, 'lr': 0.001, 'heads': 2, 'wd': 0.0005}
  [09] {'mci_weight': 3.7, 'gamma': 1.4, 'lr': 0.001, 'heads': 2, 'wd': 0.001}
  [10] {'mci_weight': 3.7, 'gamma': 1.4, 'lr': 0.001, 'heads': 4, 'wd': 0.0005}
  [11] {'mci_weight': 3.7, 'gamma': 1.4, 'lr': 0.001, 'heads': 4, 'wd': 0.001}
  [12] {'mci_weight': 3.7, '

### Data loading

In [ ]:
# data loading
print('Loading data...')

outer_folds = []
for k in range(1, N_OUTER + 1):
    outer_folds.append({
        'train': torch.load(os.path.join(DATA_DIR, f'fold_{k}_train.pt'), weights_only=False),
        'val':   torch.load(os.path.join(DATA_DIR, f'fold_{k}_val.pt'), weights_only=False),
        'test':  torch.load(os.path.join(DATA_DIR, f'fold_{k}_test.pt'), weights_only=False),
    })

with open(os.path.join(DATA_DIR, 'inner_fold_ids.pkl'), 'rb') as f:
    inner_fold_ids = pickle.load(f)

print('Done.')
for k, fold in enumerate(outer_folds):
    print(f'  Outer fold {k+1}: train={len(fold["train"])}  val={len(fold["val"])}  test={len(fold["test"])}')

Loading data...
Done.
  Outer fold 1: train=431  val=77  test=127
  Outer fold 2: train=431  val=77  test=127
  Outer fold 3: train=431  val=77  test=127
  Outer fold 4: train=431  val=77  test=127
  Outer fold 5: train=431  val=77  test=127


### Inner training loop for hyperparameter tuning

In [ ]:
# resume if training stopped midway
if os.path.exists(INNER_RESULTS_PATH):
    with open(INNER_RESULTS_PATH, 'rb') as f:
        inner_results = pickle.load(f)
    print(f'Resuming - {len(inner_results)} runs already complete.')
else:
    inner_results = {}

# creates inner split
def get_inner_split(outer_train, inner_fold_dict):
    inner_train = [g for g in outer_train if g.subject_id in inner_fold_dict['inner_train']]
    inner_val   = [g for g in outer_train if g.subject_id in inner_fold_dict['inner_val']]
    return inner_train, inner_val

# the following loop performs the hyperparamter tuning train/val phase
# it iterates over all outer and inner folds, training/validating all configurations
outer_bar = tqdm(range(N_OUTER), desc='Outer folds', position=0)
for outer_idx in outer_bar:
    outer_train = outer_folds[outer_idx]['train']

    inner_bar = tqdm(range(N_INNER), desc=f'  Inner folds', position=1, leave=False)
    for inner_idx in inner_bar:
        inner_train, inner_val = get_inner_split(outer_train, inner_fold_ids[outer_idx][inner_idx])
        train_labels = get_labels(inner_train)
        train_loader = DataLoader(inner_train, batch_size=BATCH, shuffle=True)
        val_loader   = DataLoader(inner_val,   batch_size=BATCH, shuffle=False)

        cfg_bar = tqdm(range(len(CONFIGS)), desc='    Configs', position=2, leave=False)
        for cfg_idx in cfg_bar:
            key = (outer_idx, inner_idx, cfg_idx)
            if key in inner_results:
                cfg_bar.set_postfix_str('skipped')
                continue

            cfg           = CONFIGS[cfg_idx]
            criterion     = build_criterion(cfg, train_labels)
            model, opt    = build_model_and_opt(cfg)
            _, metrics    = train_with_early_stopping(model, opt, criterion, train_loader, val_loader)

            inner_results[key] = metrics
            cfg_bar.set_postfix(bal_acc=f'{metrics["balanced_acc"]:.3f}',
                                recall_mci=f'{metrics["recall_mci"]:.3f}')

        # save after every inner fold
        with open(INNER_RESULTS_PATH, 'wb') as f:
            pickle.dump(inner_results, f)

print('Hyperparameter tuning comlpeted.')

# average metrics per config across all 15 runs
inner_avg = {}
for cfg_idx in range(len(CONFIGS)):
    runs = [inner_results[(o, i, cfg_idx)] for o in range(N_OUTER) for i in range(N_INNER)]
    inner_avg[cfg_idx] = avg_metrics(runs)

print('\nInner-loop averaged metrics per config:')
df_inner = pd.DataFrame([
    {'config': i, **inner_avg[i], **CONFIGS[i]}
    for i in range(len(CONFIGS))
]).set_index('config')
print(df_inner.to_string())
df_inner.to_csv(os.path.join(OUTPUT_DIR, 'inner_avg_metrics.csv'))

Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Inner folds:   0%|          | 0/3 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

...

  Inner folds:   0%|          | 0/3 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

Hyperparameter tuning comlpeted.

Inner-loop averaged metrics per config:
        balanced_acc  accuracy  recall_mci  recall_cn  precision_mci  precision_cn    f1_mci     f1_cn  f1_macro       auc  mci_weight  gamma     lr  heads      wd
config                                                                                                                                                             
0           0.712778  0.700236    0.733671   0.691884       0.374956      0.914265  0.491963  0.785006  0.638484  0.745612         3.7    1.0  0.001      2  0.0005
1           0.734900  0.732190    0.738244   0.731556       0.411674      0.919298  0.521867  0.812160  0.667014  0.760987         3.7    1.0  0.001      2  0.0010
2           0.708712  0.691874    0.734288   0.683136       0.373003      0.913452  0.486357  0.777432  0.631895  0.735906         3.7    1.0  0.001      4  0.0005
3           0.718645  0.710434    0.732417   0.704873       0.384579      0.915103  0.500383  0.794622  0.

### Filtering out worst configurations

In [ ]:
INNER_FILTER = {
    'balanced_acc': 0.70,
    'recall_mci':   0.70,
    'recall_cn':    0.70,
}

surviving_configs = [
    i for i in range(len(CONFIGS))
    if passes_filter(inner_avg[i], INNER_FILTER)
]

print(f'Configs surviving inner filter: {len(surviving_configs)}/{len(CONFIGS)}')
for i in surviving_configs:
    print(f'  [{i:02d}] bal_acc={inner_avg[i]["balanced_acc"]:.3f}  '
          f'recall_mci={inner_avg[i]["recall_mci"]:.3f}  '
          f'recall_cn={inner_avg[i]["recall_cn"]:.3f}  |  {CONFIGS[i]}')

Configs surviving inner filter: 11/48
  [01] bal_acc=0.735  recall_mci=0.738  recall_cn=0.732  |  {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.001, 'heads': 2, 'wd': 0.001}
  [03] bal_acc=0.719  recall_mci=0.732  recall_cn=0.705  |  {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.001, 'heads': 4, 'wd': 0.001}
  [04] bal_acc=0.728  recall_mci=0.731  recall_cn=0.725  |  {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.003, 'heads': 2, 'wd': 0.0005}
  [05] bal_acc=0.730  recall_mci=0.752  recall_cn=0.709  |  {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.003, 'heads': 2, 'wd': 0.001}
  [06] bal_acc=0.727  recall_mci=0.730  recall_cn=0.723  |  {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.003, 'heads': 4, 'wd': 0.0005}
  [12] bal_acc=0.740  recall_mci=0.749  recall_cn=0.730  |  {'mci_weight': 3.7, 'gamma': 1.4, 'lr': 0.003, 'heads': 2, 'wd': 0.0005}
  [23] bal_acc=0.723  recall_mci=0.723  recall_cn=0.722  |  {'mci_weight': 3.7, 'gamma': 1.8, 'lr': 0.003, 'heads': 4, 'wd': 0.001}
  [26] bal_acc=0.714  recall_mci=0.

### Training and validating on outer sets

In [ ]:
# surviving configurations get trained on the full train set, and validated on the outer val set
CKPT_DIR          = os.path.join(OUTPUT_DIR, 'outer_checkpoints')
OUTER_VAL_RESULTS_PATH = os.path.join(OUTPUT_DIR, 'outer_val_results.pkl')
os.makedirs(CKPT_DIR, exist_ok=True)

# resume if training stopped midway
if os.path.exists(OUTER_VAL_RESULTS_PATH):
    with open(OUTER_VAL_RESULTS_PATH, 'rb') as f:
        outer_val_results = pickle.load(f)
    print(f'Resuming - {len(outer_val_results)} runs already complete.')
else:
    outer_val_results = {}

# the following loop iterates over all outer folds, for all configurations
# each configuration gets trained on the whole train set and validated
# on the outer val set of each fold
cfg_bar = tqdm(surviving_configs, desc='Configs', position=0)
for cfg_idx in cfg_bar:
    cfg = CONFIGS[cfg_idx]

    outer_bar = tqdm(range(N_OUTER), desc=f'  Outer folds', position=1, leave=False)
    for outer_idx in outer_bar:
        key = (cfg_idx, outer_idx)
        if key in outer_val_results:
            outer_bar.set_postfix_str('skipped')
            continue

        outer_train  = outer_folds[outer_idx]['train']
        outer_val    = outer_folds[outer_idx]['val']
        train_labels = get_labels(outer_train)
        train_loader = DataLoader(outer_train, batch_size=BATCH, shuffle=True)
        val_loader   = DataLoader(outer_val,   batch_size=BATCH, shuffle=False)

        criterion    = build_criterion(cfg, train_labels)
        model, opt   = build_model_and_opt(cfg)
        model, metrics = train_with_early_stopping(model, opt, criterion, train_loader, val_loader)

        outer_val_results[key] = metrics
        outer_bar.set_postfix(bal_acc=f'{metrics["balanced_acc"]:.3f}',
                              recall_mci=f'{metrics["recall_mci"]:.3f}',
                              recall_cn=f'{metrics["recall_cn"]:.3f}')

        ckpt_path = os.path.join(CKPT_DIR, f'cfg{cfg_idx:02d}_outer{outer_idx+1}.pt')
        torch.save(model.state_dict(), ckpt_path)

        with open(OUTER_VAL_RESULTS_PATH, 'wb') as f:
            pickle.dump(outer_val_results, f)

print('Training and validation completed.')

# average outer val metrics per config across 5 folds
outer_val_avg = {
    i: avg_metrics([outer_val_results[(i, o)] for o in range(N_OUTER)])
    for i in surviving_configs
}

print('\nOuter-val averaged metrics per surviving config:')
df_val = pd.DataFrame([
    {'config': i, **outer_val_avg[i], **CONFIGS[i]}
    for i in surviving_configs
]).set_index('config')
print(df_val.to_string())
df_val.to_csv(os.path.join(OUTPUT_DIR, 'outer_val_avg_metrics.csv'))

Configs:   0%|          | 0/11 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

...

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

Training and validation completed.

Outer-val averaged metrics per surviving config:
        balanced_acc  accuracy  recall_mci  recall_cn  precision_mci  precision_cn    f1_mci     f1_cn  f1_macro       auc  mci_weight  gamma     lr  heads      wd
config                                                                                                                                                             
1           0.798648  0.820779    0.762500   0.834796       0.534444      0.936674  0.624069  0.881759  0.752914  0.811593         3.7    1.0  0.001      2  0.0010
3           0.740775  0.745455    0.734167   0.747382       0.426247      0.921556  0.535126  0.823681  0.679404  0.782727         3.7    1.0  0.001      4  0.0010
4           0.790487  0.831169    0.723333   0.857641       0.567187      0.926527  0.632071  0.890227  0.761149  0.793581         3.7    1.0  0.003      2  0.0005
5           0.792236  0.825974    0.736667   0.847805       0.554348      0.929412  0.627557  0

### Filtering out worst configurations

In [ ]:
# filtering models based on performance on outer val
OUTER_VAL_FILTER = {
    'balanced_acc': 0.75,
    'recall_mci':   0.75,
    'recall_cn':    0.725,
}

final_configs = [
    i for i in surviving_configs
    if passes_filter(outer_val_avg[i], OUTER_VAL_FILTER)
]

print(f'Configs surviving outer val filter: {len(final_configs)}/{len(surviving_configs)}')
for i in final_configs:
    print(f'  [{i:02d}] bal_acc={outer_val_avg[i]["balanced_acc"]:.3f}  '
          f'recall_mci={outer_val_avg[i]["recall_mci"]:.3f}  '
          f'recall_cn={outer_val_avg[i]["recall_cn"]:.3f}  |  {CONFIGS[i]}')

Configs surviving outer val filter: 5/11
  [01] bal_acc=0.799  recall_mci=0.762  recall_cn=0.835  |  {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.001, 'heads': 2, 'wd': 0.001}
  [12] bal_acc=0.796  recall_mci=0.855  recall_cn=0.738  |  {'mci_weight': 3.7, 'gamma': 1.4, 'lr': 0.003, 'heads': 2, 'wd': 0.0005}
  [26] bal_acc=0.768  recall_mci=0.763  recall_cn=0.774  |  {'mci_weight': 4.2, 'gamma': 1.0, 'lr': 0.001, 'heads': 4, 'wd': 0.0005}
  [28] bal_acc=0.780  recall_mci=0.790  recall_cn=0.770  |  {'mci_weight': 4.2, 'gamma': 1.0, 'lr': 0.003, 'heads': 2, 'wd': 0.0005}
  [36] bal_acc=0.760  recall_mci=0.776  recall_cn=0.745  |  {'mci_weight': 4.2, 'gamma': 1.4, 'lr': 0.003, 'heads': 2, 'wd': 0.0005}


### Testing best configurations on the test set

In [ ]:
# testing
test_results = {i: {} for i in final_configs}

# the following loop tests all configurations on the test set of all 5 outer folds
cfg_bar = tqdm(final_configs, desc='Configs', position=0)
for cfg_idx in cfg_bar:
    cfg = CONFIGS[cfg_idx]

    outer_bar = tqdm(range(N_OUTER), desc='  Outer folds', position=1, leave=False)
    for outer_idx in outer_bar:
        ckpt_path = os.path.join(CKPT_DIR, f'cfg{cfg_idx:02d}_outer{outer_idx+1}.pt')
        model, _  = build_model_and_opt(cfg)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))

        test_loader          = DataLoader(outer_folds[outer_idx]['test'], batch_size=BATCH, shuffle=False)
        labels, preds, probs = evaluate(model, test_loader)
        metrics              = compute_metrics(labels, preds, probs)

        test_results[cfg_idx][outer_idx] = {
            'metrics': metrics,
            'probs':   probs,
            'preds':   preds,
            'labels':  labels,
        }

        outer_bar.set_postfix(bal_acc=f'{metrics["balanced_acc"]:.3f}',
                              recall_mci=f'{metrics["recall_mci"]:.3f}',
                              recall_cn=f'{metrics["recall_cn"]:.3f}')

# per-config average across outer folds
test_avg = {
    i: avg_metrics([test_results[i][o]['metrics'] for o in range(N_OUTER)])
    for i in final_configs
}

print('\nTest averaged metrics per final config:')
df_test = pd.DataFrame([
    {'config': i, **test_avg[i], **CONFIGS[i]}
    for i in final_configs
]).set_index('config')
print(df_test.to_string())
df_test.to_csv(os.path.join(OUTPUT_DIR, 'test_avg_metrics.csv'))

Configs:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]


Test averaged metrics per final config:
        balanced_acc  accuracy  recall_mci  recall_cn  precision_mci  precision_cn    f1_mci     f1_cn  f1_macro       auc  mci_weight  gamma     lr  heads      wd
config                                                                                                                                                             
1           0.703199  0.733858    0.653586   0.752811       0.399734      0.900083  0.492588  0.818601  0.655595  0.740569         3.7    1.0  0.001      2  0.0010
12          0.676478  0.666142    0.694153   0.658803       0.340248      0.899202  0.452507  0.757668  0.605088  0.738550         3.7    1.4  0.003      2  0.0005
26          0.673710  0.686614    0.656503   0.690918       0.350437      0.894080  0.454217  0.777285  0.615751  0.730372         4.2    1.0  0.001      4  0.0005
28          0.659270  0.688189    0.616459   0.702082       0.349015      0.885128  0.439040  0.778975  0.609008  0.729795         4.2    1

### Making an ensemble model using the top configurations

In [ ]:
# ensemble per outer fold
# strategy: equal weighting of class probabilities across all surviving configs
print('Ensemble results per outer fold:')
ensemble_results = {}
for outer_idx in range(N_OUTER):
    labels      = test_results[final_configs[0]][outer_idx]['labels']
    n           = len(labels)
    avg_probs   = np.zeros((n, 2))
    for cfg_idx in final_configs:
        avg_probs += test_results[cfg_idx][outer_idx]['probs']
    avg_probs      /= len(final_configs)
    ensemble_preds  = avg_probs[:, 1] >= 0.5 # threshold
    metrics         = compute_metrics(labels, ensemble_preds, avg_probs)
    ensemble_results[outer_idx] = {
        **metrics,
        'labels': labels,
        'preds':  ensemble_preds,
        'probs':  avg_probs,
    }
    print(f'  Outer fold {outer_idx+1}: '
          f'bal_acc={metrics["balanced_acc"]:.3f}  '
          f'recall_mci={metrics["recall_mci"]:.3f}  '
          f'recall_cn={metrics["recall_cn"]:.3f}  '
          f'auc={metrics["auc"]:.3f}')

ensemble_avg = avg_metrics([
    {k: v for k, v in ensemble_results[o].items() if k not in ('labels', 'preds', 'probs')}
    for o in range(N_OUTER)
])
print(f'\nEnsemble average across folds:')
for k, v in ensemble_avg.items():
    print(f'  {k}: {v:.4f}')

df_ensemble = pd.DataFrame([
    {'outer_fold': o+1, **{k: v for k, v in ensemble_results[o].items() if k not in ('labels', 'preds', 'probs')}}
    for o in range(N_OUTER)
] + [{'outer_fold': 'avg', **ensemble_avg}])
df_ensemble.to_csv(os.path.join(OUTPUT_DIR, 'ensemble_results.csv'), index=False)
print(f'\nAll results saved to {OUTPUT_DIR}')

Ensemble results per outer fold:
  Outer fold 1: bal_acc=0.691  recall_mci=0.591  recall_cn=0.790  auc=0.726
  Outer fold 2: bal_acc=0.629  recall_mci=0.640  recall_cn=0.618  auc=0.702
  Outer fold 3: bal_acc=0.769  recall_mci=0.821  recall_cn=0.717  auc=0.858
  Outer fold 4: bal_acc=0.606  recall_mci=0.542  recall_cn=0.670  auc=0.669
  Outer fold 5: bal_acc=0.692  recall_mci=0.704  recall_cn=0.680  auc=0.787

Ensemble average across folds:
  balanced_acc: 0.6773
  accuracy: 0.6898
  recall_mci: 0.6595
  recall_cn: 0.6950
  precision_mci: 0.3525
  precision_cn: 0.8937
  f1_mci: 0.4584
  f1_cn: 0.7810
  f1_macro: 0.6197
  auc: 0.7483

All results saved to /content/drive/MyDrive/team3_xai_gnn/transformer_results


### Printing confusion matrices

In [ ]:
for outer_idx in range(N_OUTER):
    print(f'---Outer Fold {outer_idx+1}---')

    for cfg_idx in final_configs:
        r  = test_results[cfg_idx][outer_idx]
        cm = confusion_matrix(r['labels'], r['preds'])
        c  = CONFIGS[cfg_idx]
        print(f'cfg{cfg_idx} (mw={c["mci_weight"]} g={c["gamma"]} lr={c["lr"]} h={c["heads"]} wd={c["wd"]})')
        print(f'{"":10s}  Pred CN  Pred MCI')
        print(f'  True CN   {cm[0,0]:4d}     {cm[0,1]:4d}')
        print(f'  True MCI  {cm[1,0]:4d}     {cm[1,1]:4d}\n')

    er = ensemble_results[outer_idx]
    cm = confusion_matrix(er['labels'], er['preds'])
    print(f'ENSEMBLE')
    print(f'{"":10s}  Pred CN  Pred MCI')
    print(f'  True CN   {cm[0,0]:4d}     {cm[0,1]:4d}')
    print(f'  True MCI  {cm[1,0]:4d}     {cm[1,1]:4d}')
    print()

---Outer Fold 1---
cfg1 (mw=3.7 g=1.0 lr=0.001 h=2 wd=0.001)
            Pred CN  Pred MCI
  True CN     75       30
  True MCI     7       15

cfg12 (mw=3.7 g=1.4 lr=0.003 h=2 wd=0.0005)
            Pred CN  Pred MCI
  True CN     60       45
  True MCI     6       16

cfg26 (mw=4.2 g=1.0 lr=0.001 h=4 wd=0.0005)
            Pred CN  Pred MCI
  True CN     85       20
  True MCI    11       11

cfg28 (mw=4.2 g=1.0 lr=0.003 h=2 wd=0.0005)
            Pred CN  Pred MCI
  True CN     90       15
  True MCI    12       10

cfg36 (mw=4.2 g=1.4 lr=0.003 h=2 wd=0.0005)
            Pred CN  Pred MCI
  True CN     72       33
  True MCI     9       13

ENSEMBLE
            Pred CN  Pred MCI
  True CN     83       22
  True MCI     9       13

---Outer Fold 2---
cfg1 (mw=3.7 g=1.0 lr=0.001 h=2 wd=0.001)
            Pred CN  Pred MCI
  True CN     83       19
  True MCI    10       15

cfg12 (mw=3.7 g=1.4 lr=0.003 h=2 wd=0.0005)
            Pred CN  Pred MCI
  True CN     58       44
  True MCI  